# Multi-Phase Bayesian Optimization for MOF Synthesis

This notebook demonstrates the **autonomous multi-phase Bayesian optimization** system for ZIF synthesis.

## Overview

The system runs through **3 phases**:
1. **Exploration** (batch=10, xi=0.05): Broad screening of composition space
2. **Refinement** (batch=5, xi=0.01): Focused optimization of promising regions
3. **Validation** (batch=2, xi=0.001): Fine-tuning around optimum

## Features

- **Approval Modes**: Manual (interactive prompts) or Automatic (fully autonomous)
- **State Persistence**: Resume capability with full phase state
- **Dynamic Bounds**: Search space progressively narrows to best regions
- **Automatic Transitions**: Based on iteration count, performance, and EI convergence

---
## Part 1: Setup and Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add src to path
sys.path.insert(0, str(Path.cwd() / "src"))

from science_jubilee.Machine import Machine
from science_jubilee.Experiment import Experiment
from science_jubilee.tools.Syringe import Syringe
from science_jubilee.tools.Double_Syringe import DoubleSyringe
from science_jubilee.tools.Vacuum_Gripper import VacuumGripper
from science_jubilee.tools.Oceandirect_axo import Spectrometer

from science_jubilee.optimization.orchestrator import BayesianOptimizationLoop
from science_jubilee.optimization.phase_config import (
    PhaseConfig, DEFAULT_PHASES, create_single_phase_config
)
from science_jubilee.optimization.phase_manager import PhaseManager

print("✓ All imports successful")

---
## Part 2: Configuration

Choose your optimization mode and settings.

In [ ]:
# =============================================================================
# CONFIGURATION SETTINGS
# =============================================================================

# Machine IP address
JUBILEE_IP = "192.168.1.2"

# Operator information
OPERATOR_NAME = "YourName"  # Change this!

# Output directory for results
OUTPUT_DIR = "optimization_results"

# =============================================================================
# MULTI-PHASE SETTINGS
# =============================================================================

# Approval mode: "manual" or "automatic"
# - manual: Prompt for approval at each phase transition (recommended for first run)
# - automatic: Fully autonomous, no prompts
APPROVAL_MODE = "manual"

# Phase configuration: "multi" or "single"
# - multi: 3-phase optimization (exploration → refinement → validation)
# - single: Traditional BO with fixed batch size
PHASE_MODE = "multi"

# For single-phase mode only:
SINGLE_PHASE_BATCH_SIZE = 5
SINGLE_PHASE_MAX_ITERATIONS = 20

# Number of initial samples (for new runs)
N_INITIAL_SAMPLES = 5

# =============================================================================
# RESUME SETTINGS
# =============================================================================

# Set to True to resume from saved state
RESUME_FROM_SAVED_STATE = False

# State file location
STATE_FILE = f"{OUTPUT_DIR}/bo_state.json"

# =============================================================================
# DISPLAY CONFIGURATION
# =============================================================================

print("Configuration:")
print(f"  Jubilee IP: {JUBILEE_IP}")
print(f"  Operator: {OPERATOR_NAME}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Approval mode: {APPROVAL_MODE}")
print(f"  Phase mode: {PHASE_MODE}")
if PHASE_MODE == "multi":
    print(f"  Phases:")
    for i, phase in enumerate(DEFAULT_PHASES, 1):
        print(f"    {i}. {phase.name}: batch_size={phase.batch_size}, xi={phase.xi}, iterations={phase.min_iterations}-{phase.max_iterations}")
else:
    print(f"  Batch size: {SINGLE_PHASE_BATCH_SIZE}")
    print(f"  Max iterations: {SINGLE_PHASE_MAX_ITERATIONS}")
print(f"  Resume: {RESUME_FROM_SAVED_STATE}")
print(f"  Initial samples: {N_INITIAL_SAMPLES}")

---
## Part 3: Hardware Initialization

### ⚠️ IMPORTANT: Manual Calibration Required

Before running this section, ensure you have:
1. **Removed labware from slots 0-1** (prevents collision during homing)
2. **Completed manual offset calibration** for all labware
3. **Saved offset files** in the labware definition directories

See `mof_synthesis.ipynb` for manual calibration procedure.

In [ ]:
# =============================================================================
# STEP 1: Initialize Machine
# =============================================================================

print("Connecting to Jubilee...")
machine = Machine(address=JUBILEE_IP)
print(f"✓ Connected to {JUBILEE_IP}")

print("\nHoming all axes...")
machine.home_all()
machine.move_to(z=200)  # Move to safe Z height
print("✓ Homing complete")

In [ ]:
# =============================================================================
# STEP 2: Load Deck
# =============================================================================

print("Loading deck configuration...")
deck = machine.load_deck("lab_automation_deck")
print("✓ Deck loaded")

In [ ]:
# =============================================================================
# STEP 3: Load Labware
# =============================================================================

print("Loading labware...\n")

# Slot 1: Precursors (metal + organic)
print("  Slot 1: Precursors")
precursors = machine.load_labware(
    'uwsunlab_2_wellplate_60000ul_slot1.json',
    slot=1,
    has_lid_on_top=False,
    currentLiquidVolume=60  # 60 mL initial volume
)
precursors.load_manualOffset()
print("    ✓ Precursors loaded with manual offsets")

# Slot 3: Solvents
print("  Slot 3: Solvents")
solvents = machine.load_labware(
    'uwsunlab_2_wellplate_60000ul_slot3.json',
    slot=3,
    has_lid_on_top=False,
    currentLiquidVolume=60
)
solvents.load_manualOffset()
print("    ✓ Solvents loaded with manual offsets")

# Slot 2: Sample vials (10-well plate, 3 views)
print("  Slot 2: Sample vials (3 views)")
samples2_ssy = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_ssy_1.json',
    slot=2,
    has_lid_on_top=False
)
samples2_ssy.load_manualOffset()

samples2_sy = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_sy_1.json',
    slot=2,
    has_lid_on_top=False
)
samples2_sy.load_manualOffset()

samples2_spec = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_spec_1.json',
    slot=2,
    has_lid_on_top=False
)
samples2_spec.load_manualOffset()
print("    ✓ Slot 2 samples loaded (ssy, sy, spec views)")

# Slot 5: Additional sample vials (10-well plate, 3 views)
print("  Slot 5: Additional sample vials (3 views)")
samples5_ssy = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_ssy_2.json',
    slot=5,
    has_lid_on_top=False
)
samples5_ssy.load_manualOffset()

samples5_sy = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_sy_2.json',
    slot=5,
    has_lid_on_top=False
)
samples5_sy.load_manualOffset()

samples5_spec = machine.load_labware(
    'uwsunlab_10_wellplate_14000ul_spec_2.json',
    slot=5,
    has_lid_on_top=False
)
samples5_spec.load_manualOffset()
print("    ✓ Slot 5 samples loaded (ssy, sy, spec views)")

print("\n✓ All labware loaded")

In [ ]:
# =============================================================================
# STEP 4: Load Tools
# =============================================================================

print("Loading tools...\n")

# Single syringe (T0)
print("  T0: Single Syringe")
single_syringe = Syringe(
    index=0,
    name='single_syringe',
    config='single_syringe'
)
machine.load_tool(single_syringe)
print("    ✓ Single syringe loaded")

# Dual syringe (T2)
print("  T2: Dual Syringe")
dual_syringe = DoubleSyringe(
    index=2,
    name='Dual_Syringe',
    config='10cc_syringe'
)
machine.load_tool(dual_syringe)
print("    ✓ Dual syringe loaded")

# Spectrometer (T3)
print("  T3: Spectrometer")
spectrometer = Spectrometer(
    index=3,
    name='Spectrometer',
    base_dir=r"C:\Axo\science-jubilee\axo\spectrum_data",
    experiment_name='BO_Optimization',  # Will be overwritten by orchestrator
    operator_name=OPERATOR_NAME,
    target_compound='ZIF',
    experiment_notes='Autonomous multi-phase Bayesian optimization',
    solvent='triethylamine/methanol',
    temperature_c=25
)
machine.load_tool(spectrometer)
print("    ✓ Spectrometer loaded")

# Vacuum gripper (T4)
print("  T4: Vacuum Gripper")
gripper = VacuumGripper(
    index=4,
    name='Vacuum_Gripper',
    vacuum_pin=0,
    limit_switch_pin=2
)
machine.load_tool(gripper)
print("    ✓ Vacuum gripper loaded")

print("\n✓ All tools loaded")

In [ ]:
# =============================================================================
# STEP 5: Configure Vacuum Gripper Locations
# =============================================================================

vacuum_location = {
    0: {"loc": (84, 51, 0), "labwares_list": []},
    1: {"loc": (225, 57, 0), "labwares_list": [precursors]},
    2: {"loc": (84, 147, 0), "labwares_list": [samples2_ssy, samples2_sy, samples2_spec]},
    3: {"loc": (225, 149, 0), "labwares_list": [solvents]},
    4: {"loc": (79, 244, 0), "labwares_list": []},
    5: {"loc": (221, 244, 0), "labwares_list": [samples5_ssy, samples5_sy, samples5_spec]}
}

print("✓ Vacuum gripper locations configured")

In [ ]:
# =============================================================================
# STEP 6: Organize Labware and Tools for Experiment
# =============================================================================

all_labwares = {
    "slot1": {"precursors": precursors},
    "slot2": {
        "samples2_ssy": samples2_ssy,
        "samples2_sy": samples2_sy,
        "samples2_spec": samples2_spec
    },
    "slot3": {"solvents": solvents},
    "slot4": {"vacuum_location": vacuum_location},
    "slot5": {
        "samples5_ssy": samples5_ssy,
        "samples5_sy": samples5_sy,
        "samples5_spec": samples5_spec
    }
}

all_tools = {
    "single_syringe": single_syringe,
    "dual_syringe": dual_syringe,
    "spectrometer": spectrometer,
    "gripper": gripper
}

print("✓ Labware and tools organized")

In [ ]:
# =============================================================================
# STEP 7: Create Experiment Instance
# =============================================================================

print("Creating Experiment instance...")
experiment = Experiment(
    machine=machine,
    deck=deck,
    all_tools=all_tools,
    all_labwares=all_labwares
)

print("✓ Experiment instance created")
print("\n" + "="*70)
print("HARDWARE SETUP COMPLETE")
print("="*70)

---
## Part 4: Initialize Orchestrator

Create the multi-phase Bayesian optimization orchestrator.

In [ ]:
# =============================================================================
# STEP 8: Configure Phases
# =============================================================================

if PHASE_MODE == "multi":
    phases = DEFAULT_PHASES
    print("Using multi-phase configuration:")
    for i, phase in enumerate(phases, 1):
        print(f"  Phase {i}: {phase.name}")
        print(f"    Batch size: {phase.batch_size}")
        print(f"    Exploration (xi): {phase.xi}")
        print(f"    Iterations: {phase.min_iterations}-{phase.max_iterations}")
        print(f"    Description: {phase.description}")
else:
    phases = create_single_phase_config(
        batch_size=SINGLE_PHASE_BATCH_SIZE,
        max_iterations=SINGLE_PHASE_MAX_ITERATIONS,
        xi=0.01
    )
    print("Using single-phase configuration:")
    print(f"  Batch size: {SINGLE_PHASE_BATCH_SIZE}")
    print(f"  Max iterations: {SINGLE_PHASE_MAX_ITERATIONS}")

In [ ]:
# =============================================================================
# STEP 9: Create Orchestrator
# =============================================================================

print("\nCreating Bayesian Optimization Orchestrator...")

orchestrator = BayesianOptimizationLoop(
    machine=machine,
    experiment=experiment,
    phases=phases,
    approval_mode=APPROVAL_MODE,
    output_dir=OUTPUT_DIR,
    n_initial_samples=N_INITIAL_SAMPLES,
    convergence_threshold=0.001,
    state_file="bo_state.json",
    operator_name=OPERATOR_NAME
)

print("✓ Orchestrator created")
print(f"  Output directory: {orchestrator.output_dir}")
print(f"  Current phase: {orchestrator.phase_manager.current_phase.name}")
print(f"  Approval mode: {orchestrator.phase_manager.approval_mode}")
print(f"  State file: {orchestrator.state_file}")

---
## Part 5: Resume from Saved State (Optional)

**Skip this section if starting a new run.**

If you're resuming from a previous run, execute this cell.

In [ ]:
# =============================================================================
# OPTIONAL: Resume from Saved State
# =============================================================================

if RESUME_FROM_SAVED_STATE:
    print("Loading saved state...")
    orchestrator.load_state()
    
    print("\nResumed state:")
    print(f"  Iteration: {orchestrator.iteration}")
    print(f"  Total samples: {len(orchestrator.X) if orchestrator.X is not None else 0}")
    print(f"  Current phase: {orchestrator.phase_manager.current_phase.name}")
    print(f"  Phase iteration: {orchestrator.phase_manager.phase_iteration}")
    print(f"  Approval mode: {orchestrator.phase_manager.approval_mode}")
    
    if orchestrator.y is not None and len(orchestrator.y) > 0:
        best_idx = int(np.argmax(orchestrator.y))
        print(f"  Best yield so far: {orchestrator.y[best_idx]:.4f}")
        print(f"  Best composition: Co={orchestrator.X[best_idx, 0]:.1f}, MIM={orchestrator.X[best_idx, 1]:.1f}, TEA={orchestrator.X[best_idx, 2]:.1f}")
else:
    print("Starting new optimization run (no state loaded)")

---
## Part 6: Run Optimization Loop

### ⚠️ IMPORTANT: What to Expect

The optimization loop will:

1. **Generate synthesis plans** for each batch
2. **Prompt you to load vials** (you must press ENTER to continue)
3. **Run automated synthesis** (mixing, spectrum collection)
4. **Extract yields** from spectral data using Gualtieri fitting
5. **Update GP model** and select next batch
6. **Check for phase transitions** (if multi-phase mode)
7. **Prompt for approval** (if manual approval mode)
8. **Repeat** until convergence

### Manual Approval Mode

When phase transition criteria are met, you'll see:
```
======================================================================
PHASE TRANSITION PROPOSED
======================================================================
Current Phase: EXPLORATION
...
Approve transition to 'refinement' phase? [Y/n/info]: 
```

Options:
- **Y/yes/ENTER**: Approve transition
- **n/no**: Reject transition (continue current phase)
- **info**: Display detailed phase information

### Interrupting the Run

- Press **Kernel → Interrupt** to safely stop
- State will be saved automatically
- Resume later by setting `RESUME_FROM_SAVED_STATE = True` and re-running from Part 5

In [ ]:
# =============================================================================
# RUN THE OPTIMIZATION LOOP
# =============================================================================

print("="*70)
print("STARTING MULTI-PHASE BAYESIAN OPTIMIZATION")
print("="*70)
print(f"Approval mode: {APPROVAL_MODE}")
print(f"Phase mode: {PHASE_MODE}")
print()

# Run the autonomous loop
orchestrator.run()

print("\n" + "="*70)
print("OPTIMIZATION COMPLETE!")
print("="*70)

---
## Part 7: Results Analysis

Analyze the optimization results.

In [ ]:
# =============================================================================
# VIEW FINAL RESULTS
# =============================================================================

if orchestrator.X is not None and orchestrator.y is not None:
    print("FINAL RESULTS")
    print("="*70)
    
    # Best result
    best_idx = int(np.argmax(orchestrator.y))
    best_yield = orchestrator.y[best_idx]
    best_comp = orchestrator.X[best_idx]
    
    print(f"\nOptimal Conditions:")
    print(f"  Co²⁺ volume: {best_comp[0]:.2f} mL")
    print(f"  2-methylimidazole volume: {best_comp[1]:.2f} mL")
    print(f"  Triethylamine volume: {best_comp[2]:.2f} mL")
    print(f"  Best yield (I_max): {best_yield:.4f}")
    
    # Summary statistics
    print(f"\nSummary:")
    print(f"  Total experiments: {len(orchestrator.y)}")
    print(f"  Total iterations: {orchestrator.iteration}")
    print(f"  Mean yield: {np.mean(orchestrator.y):.4f}")
    print(f"  Std yield: {np.std(orchestrator.y):.4f}")
    print(f"  Final phase: {orchestrator.phase_manager.current_phase.name}")
    
    # Convergence info
    if orchestrator.converged:
        print(f"\nConvergence:")
        print(f"  Status: CONVERGED")
        print(f"  Reason: {orchestrator.convergence_reason}")
    
    print(f"\nResults saved to: {orchestrator.output_dir}")
    print(f"State file: {orchestrator.state_file}")
    
else:
    print("No results available yet (optimization not started or no data collected)")

In [ ]:
# =============================================================================
# PLOT OPTIMIZATION PROGRESS
# =============================================================================

if orchestrator.X is not None and orchestrator.y is not None and len(orchestrator.y) > 0:
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Yield over iterations
    ax = axes[0, 0]
    iterations = np.arange(len(orchestrator.y))
    best_so_far = np.maximum.accumulate(orchestrator.y)
    
    ax.scatter(iterations, orchestrator.y, alpha=0.5, label='Individual yields')
    ax.plot(iterations, best_so_far, 'r-', linewidth=2, label='Best yield so far')
    ax.set_xlabel('Experiment Number')
    ax.set_ylabel('Yield (I_max)')
    ax.set_title('Optimization Progress')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: EI history
    ax = axes[0, 1]
    if len(orchestrator.ei_history) > 0:
        ax.plot(orchestrator.ei_history, 'o-', linewidth=2)
        ax.axhline(y=orchestrator.convergence_threshold, color='r', linestyle='--', 
                   label=f'Convergence threshold ({orchestrator.convergence_threshold})')
        ax.set_xlabel('Iteration')
        ax.set_ylabel('Max Expected Improvement')
        ax.set_title('Expected Improvement Over Time')
        ax.set_yscale('log')
        ax.legend()
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No EI history available', 
                ha='center', va='center', transform=ax.transAxes)
    
    # Plot 3: Composition space (Co vs MIM)
    ax = axes[1, 0]
    scatter = ax.scatter(orchestrator.X[:, 0], orchestrator.X[:, 1], 
                        c=orchestrator.y, cmap='viridis', s=100, alpha=0.7)
    ax.scatter(best_comp[0], best_comp[1], c='red', s=300, marker='*', 
              edgecolors='white', linewidth=2, label='Best', zorder=10)
    ax.set_xlabel('Co²⁺ (mL)')
    ax.set_ylabel('2-methylimidazole (mL)')
    ax.set_title('Composition Space: Co vs MIM')
    plt.colorbar(scatter, ax=ax, label='Yield')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 4: Composition space (Co vs TEA)
    ax = axes[1, 1]
    scatter = ax.scatter(orchestrator.X[:, 0], orchestrator.X[:, 2], 
                        c=orchestrator.y, cmap='viridis', s=100, alpha=0.7)
    ax.scatter(best_comp[0], best_comp[2], c='red', s=300, marker='*', 
              edgecolors='white', linewidth=2, label='Best', zorder=10)
    ax.set_xlabel('Co²⁺ (mL)')
    ax.set_ylabel('Triethylamine (mL)')
    ax.set_title('Composition Space: Co vs TEA')
    plt.colorbar(scatter, ax=ax, label='Yield')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    output_fig = Path(orchestrator.output_dir) / "optimization_summary.png"
    plt.savefig(output_fig, dpi=300, bbox_inches='tight')
    print(f"\nFigure saved to: {output_fig}")
    
    plt.show()
    
else:
    print("No data available for plotting")

In [ ]:
# =============================================================================
# VIEW PHASE PROGRESSION (Multi-Phase Only)
# =============================================================================

if PHASE_MODE == "multi" and orchestrator.phase_manager.phase_start_samples:
    print("PHASE PROGRESSION")
    print("="*70)
    
    phase_names = [p.name for p in orchestrator.phase_manager.phases]
    phase_starts = orchestrator.phase_manager.phase_start_samples
    
    for i, (name, start) in enumerate(zip(phase_names, phase_starts)):
        if i < len(phase_starts) - 1:
            end = phase_starts[i + 1]
            n_samples = end - start
        else:
            n_samples = len(orchestrator.y) - start if orchestrator.y is not None else 0
        
        print(f"\nPhase {i+1}: {name.upper()}")
        print(f"  Samples: {start} to {start + n_samples - 1} ({n_samples} total)")
        
        if orchestrator.y is not None and n_samples > 0:
            phase_yields = orchestrator.y[start:start + n_samples]
            print(f"  Best yield in phase: {np.max(phase_yields):.4f}")
            print(f"  Mean yield in phase: {np.mean(phase_yields):.4f}")
            print(f"  Improvement from previous: {(np.max(phase_yields) - np.max(orchestrator.y[:start])) if start > 0 else 0:.4f}")

else:
    print("Phase progression not available (single-phase mode or no data)")

---
## Part 8: Cleanup and Shutdown

Safely shutdown the system.

In [ ]:
# =============================================================================
# CLEANUP
# =============================================================================

print("Cleaning up...")

# Park current tool
try:
    machine.park_tool()
    print("✓ Tool parked")
except:
    print("  (No tool was active)")

# Move to safe position
machine.move_to(z=200)
machine.move_to(x=0, y=0)
print("✓ Moved to home position")

# Save final state
orchestrator.save_state()
print(f"✓ Final state saved to {orchestrator.state_file}")

print("\n" + "="*70)
print("CLEANUP COMPLETE")
print("="*70)
print(f"\nAll results saved to: {orchestrator.output_dir}")
print("Check the following files:")
print(f"  - {orchestrator.state_file} (resume state)")
print(f"  - {orchestrator.output_dir}/bo_optimization.log (detailed log)")
print(f"  - {orchestrator.output_dir}/plots/ (visualization plots)")
print(f"  - {orchestrator.output_dir}/synthesis_plan_*.json (experiment plans)")

---
## Troubleshooting

### Common Issues

1. **"Error: machine must first be homed"**
   - Solution: Re-run Part 3, Step 1 (machine.home_all())

2. **"Well cannot accommodate X ml dispense liquid volume"**
   - Solution: Check currentLiquidVolume initialization in Part 3, Step 3

3. **"Lid is on top of wellPlate"**
   - Solution: Set `labware.has_lid_on_top = False` manually

4. **Phase transition not occurring**
   - Check: Min iterations met? Performance threshold met? EI converged?
   - Review logs in `{OUTPUT_DIR}/bo_optimization.log`

5. **Keyboard interrupt during run**
   - State is automatically saved
   - Set `RESUME_FROM_SAVED_STATE = True` and restart from Part 5

### Getting Help

- Check detailed logs: `{OUTPUT_DIR}/bo_optimization.log`
- Review CLAUDE.md for system architecture
- Review MULTI_PHASE_IMPLEMENTATION.md for phase system details